# Diabetes Prediction Using Artificial Intelligence
### Educational Capstone Project - AI/ML Internship

This Jupyter Notebook demonstrates the complete pipeline developed during the 15-day Artificial Intelligence internship. It covers Python basics, NumPy, Pandas, Matplotlib, Preprocessing, Feature Engineering, Regression, Classification, Unsupervised Clustering, and Deep Learning (Artificial Neural Networks).

**Educational Disclaimer:** This project is developed for Artificial Intelligence internship learning and demonstration purposes only. The predictions generated by these machine learning models are not medical diagnoses. Please consult a qualified healthcare professional for medical advice.

## Day 2: Environment Setup and Data Loading
We start by importing our core libraries: NumPy, Pandas, Matplotlib, Scikit-Learn, and TensorFlow.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, r2_score, mean_absolute_error
from sklearn.cluster import KMeans, AgglomerativeClustering
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("TensorFlow version:", tf.__version__)
print("Scikit-Learn version:", pd.__version__)

### Loading the Dataset
We use Pandas to load the raw CSV file.

In [ ]:
# Load the diabetes dataset
df = pd.read_csv('../data/diabetes.csv')
print(f'Dataset loaded successfully with shape: {df.shape}')
df.head()

## Day 3: Data Inspection & Pandas Exploration
We inspect the details of the dataset, including rows, columns, data types, summary statistics, missing values, and target outcomes.

In [ ]:
# 1. Display column data types
print('--- Column Data Types ---')
print(df.dtypes)

# 2. Summary statistics
print('\n--- Statistical Summary ---')
print(df.describe())

# 3. Null values count
print('\n--- Missing Values Count ---')
print(df.isnull().sum())

# 4. Class balance
print('\n--- Target (Outcome) Distribution ---')
class_counts = df['Outcome'].value_counts()
print(class_counts)

## Day 3 (Continued): Data Visualization using Matplotlib
Let's visualize the target classes and feature distributions to understand the clinical indicators.

In [ ]:
# Plot class distribution
plt.figure(figsize=(5, 4))
plt.bar(['Non-Diabetic (0)', 'Diabetic (1)'], class_counts.values, color=['#7B61FF', '#00D1FF'], width=0.5)
plt.title('Outcome Class Distribution')
plt.ylabel('Count')
plt.show()

# Plot distributions of key metrics
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
cols = [('Glucose', '#00D1FF'), ('BMI', '#7B61FF'), ('Age', '#FF6B6B'), ('BloodPressure', '#FFD166')]
for idx, (col, color) in enumerate(cols):
    ax = axes[idx // 2, idx % 2]
    valid_data = df[df[col] > 0][col]
    ax.hist(valid_data, bins=20, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()

## Day 5: Data Preprocessing & Feature Engineering
### Data Cleaning: Zeros to NaN
In physical terms, features like Glucose, BloodPressure, SkinThickness, Insulin, and BMI cannot be zero. These zero values represent invalid or missing observations. We replace them with `np.nan` so they can be imputed properly.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print('Missing value count after replacing zero values with NaN:')
print(df_clean.isnull().sum())

### Imputation and Split
We split our clean dataset into features `X` (8 columns) and label `y` (Outcome). We perform a train-test split, then fit a `SimpleImputer` on the training set to impute the missing values with column medians.

In [ ]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='median')
X_train_imputed_arr = imputer.fit_transform(X_train)
X_test_imputed_arr = imputer.transform(X_test)

X_train_imputed = pd.DataFrame(X_train_imputed_arr, columns=X.columns)
X_test_imputed = pd.DataFrame(X_test_imputed_arr, columns=X.columns)

print('Imputation complete. Null count in imputed train set:', X_train_imputed.isnull().sum().sum())

### Feature Engineering
We create the interactive feature `Glucose_Age_Interaction = Glucose * Age` on the imputed features.

In [ ]:
X_train_imputed['Glucose_Age_Interaction'] = X_train_imputed['Glucose'] * X_train_imputed['Age']
X_test_imputed['Glucose_Age_Interaction'] = X_test_imputed['Glucose'] * X_test_imputed['Age']

print('Feature engineered. Columns now:', list(X_train_imputed.columns))
X_train_imputed.head()

### Feature Scaling
We scale our 9 features using `StandardScaler` to ensure zero mean and unit variance, preparing the inputs for KNN and Neural Networks.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print('Scaling complete. Mean of first feature:', X_train_scaled[:, 0].mean())
print('Variance of first feature:', X_train_scaled[:, 0].var())

## Day 6: Educational Regression Demonstration
Since diabetes prediction is a binary classification problem, we demonstrate Simple and Multiple Linear Regression separately on a continuous variable (predicting `BloodPressure` using `BMI` and `Age`).

In [ ]:
# Load subset of continuous data
reg_data = df[['BloodPressure', 'BMI', 'Age', 'Glucose']].copy().replace(0, np.nan).dropna()
y_reg = reg_data['BloodPressure']

# Simple Regression
X_simple = reg_data[['BMI']]
reg_simple = LinearRegression().fit(X_simple, y_reg)
print('--- Simple Linear Regression ---')
print(f'Formula: BloodPressure = {reg_simple.coef_[0]:.4f} * BMI + {reg_simple.intercept_:.4f}')
print(f'Score (R2): {reg_simple.score(X_simple, y_reg):.4f}')

# Multiple Regression
X_multi = reg_data[['BMI', 'Age', 'Glucose']]
reg_multi = LinearRegression().fit(X_multi, y_reg)
print('\n--- Multiple Linear Regression ---')
print('Coefficients:', dict(zip(X_multi.columns, reg_multi.coef_)))
print(f'Intercept: {reg_multi.intercept_:.4f}')
print(f'Score (R2): {reg_multi.score(X_multi, y_reg):.4f}')

## Day 6 (Continued): Classification
We train our three classification algorithms on the scaled training features:
1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Trees

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)

# Train KNN (K=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Train Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_scaled, y_train)

print('All classifiers trained successfully.')

## Day 7: Model Evaluation (Accuracy, Precision, Recall, Confusion Matrix)
Let's evaluate each classification model on the unseen test set.

In [ ]:
classifiers = {'Logistic Regression': lr, 'K-Nearest Neighbors': knn, 'Decision Tree': dt}

for name, model in classifiers.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f'\n=== {name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'Confusion Matrix:')
    print(f'  TN: {tn} | FP: {fp}')
    print(f'  FN: {fn} | TP: {tp}')

## Day 7 (Continued): Unsupervised Clustering (K-Means & Hierarchical)
We demonstrate unsupervised grouping using Glucose and BMI as coordinates. We configure K=3 clusters.

In [ ]:
# Impute Glucose & BMI for clustering
clust_data = df[['Glucose', 'BMI']].copy().replace(0, np.nan)
clust_imputed = SimpleImputer(strategy='median').fit_transform(clust_data)
clust_scaled = StandardScaler().fit_transform(clust_imputed)

# K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(clust_scaled)

# Hierarchical (Agglomerative)
hier = AgglomerativeClustering(n_clusters=3)
hier_labels = hier.fit_predict(clust_scaled)

print('Clustering completed. Visual representation available on the Clustering page.')

## Day 8: Deep Learning & Artificial Neural Network (ANN)
We build and train a Feedforward Neural Network using Keras, documenting hidden layers, forward propagation, and backpropagation.

In [ ]:
# Build the ANN
ann = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],), name='hidden1'),
    Dense(8, activation='relu', name='hidden2'),
    Dense(1, activation='sigmoid', name='output')
])

ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann.summary()

# Train
history = ann.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs=20, batch_size=32, verbose=1)

### Evaluate the ANN

In [ ]:
ann_prob = ann.predict(X_test_scaled).flatten()
ann_pred = (ann_prob > 0.5).astype(int)

print(f'=== Artificial Neural Network ===')
print(f'Accuracy:  {accuracy_score(y_test, ann_pred):.4f}')
print(f'Precision: {precision_score(y_test, ann_pred):.4f}')
print(f'Recall:    {recall_score(y_test, ann_pred):.4f}')

## Conclusion & Model Selection
We compare our evaluation results to select the optimal model for deploying to production. This completes the Jupyter Notebook pipeline demonstration.